# Pairwise alignments

---
## Learning Objectives

1. Conceptually undersand dynamic programming and sequence alignment
1. Implement Smith-Waterman algorithm for local alignment


---
## Background

This week we will be implementing Smith-Waterman. This is a dynamic programming algorithm used for local sequence alignment. 

As a reminder, the scoring for Smith-Waterman only uses the scores from the positions above, left, and above-left of the current position in the matrix as below:

<center><img src="figures/Smith-Waterman_scoring.png"></center>

For traceback, you will need to keep track of the direction of the arrows in a matrix and then begin traceback from the maximum value.

# Smith-Waterman Algorithm Implementation Activity
## Introduction

Pairwise sequence alignment is a fundamental technique in bioinformatics used to compare two biological sequences, such as DNA, RNA, or proteins. It helps identify regions of similarity that may indicate functional, structural, or evolutionary relationships between the sequences.

The brute force method for sequence alignment, which involves comparing every possible alignment, is highly inefficient. For two sequences of length m and n, the time complexity would be \(T(m,n) = O(mn \cdot 2^{m+n})\), making it impractical for longer sequences.

Dynamic programming offers a more efficient solution. It breaks down the problem into smaller subproblems and stores their solutions to avoid redundant computations. This approach reduces the time complexity to \(O(mn)\).

The Smith-Waterman algorithm is a dynamic programming approach for local sequence alignment. It identifies the optimal local alignment between two sequences by comparing all possible pairs of segments from the sequences and finding the best-scoring alignment.

## Activity Overview

In this activity, you will implement the Smith-Waterman algorithm for local sequence alignment. The algorithm consists of two main parts:

Scoring matrix calculation
Traceback for optimal alignment reconstruction

The pseudocode for the Smith-Waterman algorithm is as follows:

$$
\begin{aligned}
& \text{Initialize scoring matrix } H \text{ with zeros} \\
& \text{For } i \text{ from } 1 \text{ to } m: \\
&     \text{For } j \text{ from } 1 \text{ to } n: \\
&         H[i][j] = \max \begin{cases}
&             0 \\
&             H[i-1][j-1] + \text{match/mismatch score} \\
&             H[i-1][j] + \text{gap penalty} \\
&             H[i][j-1] + \text{gap penalty}
&         \end{cases} \\
& \text{Find the highest score in } H \text{ and its position} \\
& \text{Perform traceback from the highest score position} \\
& \text{Return the optimal local alignment}
\end{aligned}
$$

## Instructions

Working in pairs, implement the Smith-Waterman algorithm using Python. Your implementation should include:

1. A function to calculate the scoring matrix (`cal_score`)
2. A function to perform the traceback and reconstruct the optimal alignment (`traceback`)
3. A dynamic scoring strategy that allows for customizable gap penalties and match/mismatch scores (`smith_waterman`)

## Follow these steps:

### Implement the scoring matrix calculation function:

1. Initialize the matrix with zeros
2. Fill the matrix using the Smith-Waterman algorithm
3. Return the completed matrix and the position of the highest score

### Implement the traceback function

1. Start from the highest score position
2. Trace back through the matrix to reconstruct the optimal alignment
3. Return the aligned sequences and the alignment score

### Implement a main function that:

1. Takes two sequences as input
2. Accepts parameters for match score, mismatch penalty, and gap penalty
3. Calls the scoring and traceback functions
4. Prints the aligned sequences and the alignment score

Test your implementation with various input sequences and scoring parameters

---
## Imports

In [9]:
import numpy as np
import pandas as pd

---
## Implement Smith-Waterman algorithm


```
SmithWaterman(seq1, seq2, match, mismatch, gap)
    Initialize [len(seq1)+1] x [len(seq2)+1] numpy array as scoring matrix with first column and row equal to 0
    
    Fill scoring matrix (score_matrix) and Traceback matrix, record the position with max score (max_pos):
    for i in each row number:
        for j in each column number:
            S[i][j] = max( S[i-1][j-1] + compute_diag_score, S[i-1][j] + gap_score, H[i][j-1] + gap_score, 0 )
            T[i][j] = direction of max( S[i-1][j-1] + compute_diag_score, S[i-1][j] + gap_score, S[i][j-1] + gap_score, 0 )
    
    Traceback. Find the optimal path through scoring matrix starting at max_pos
    
```

In [10]:
def cal_score(matrix, seq1, seq2, i, j, match, mismatch, gap):
    '''Calculate score for position (i,j) in scoring matrix, also record move to trace back
    
    Args:
        matrix (numpy array): scoring matrix
        seq1 (str): sequence 1 (vertical)
        seq2 (str): sequence 2 (horizontal)
        i (int): current position in seq 1's string
        j (int): current position in seq 2's string
        *NOTE*: i and j are 1 lower than they should be, this makes accessing the strings 
        and the diagonals easier to look at, while making the left and up a little wack
        match (int): score to add to the diagonal if it's a match
        mismatch (int): score to add to the diagonal if it's a mismatch (MAKE IT NEGATIVE)
        gap (int): score to add to the left and up scores (MAKE IT NEGATIVE)


    Returns:
        score in position (i,j)    
        move to trace back: 0-END, 1-DIAG, 2-UP, 3-LEFT
        
    Pseudocode:
        Calculate scores based on upper-left, up, and left neighbors:
            diag_score = upper-left + (match or mismatch)
            up_score = up + gap
            left_score = left + gap
        score = max(0, diag_score, up_score, left_score)
        traceback = maximum direction or end
        
    '''
    # initialize a dict mapping traceback directions to the ints that will represent them
    traceback_encoding = {"end": 0, "diag": 1, "up": 2, "left": 3}

    # access the left, above, and diag positions
    diag_score = matrix[i,j]
    left_score = matrix[i+1, j] + gap
    up_score = matrix[i, j+1] + gap

    # check for a match
    match_score = match if seq1[i] == seq2[j] else mismatch
    diag_score += match_score

    # make a dict for evil
    directions = {"up": up_score, "left": left_score, "diag": diag_score, "end": 0}
    
    # calculate our actual score at this position as the maximum
    real_score = max(directions.values())

    # now check which direction our score came from
    for direction, curr_score in directions.items():
        if real_score == curr_score:
            # this is the direction the score came from, so ship it
            return real_score, traceback_encoding[direction]

In [3]:
def traceback(seq1, seq2, traceback_matrix, maximum_position):
    '''Find the optimal path through scoring marix
        
        diagonal: match/mismatch
        up: gap in seq1
        left: gap in seq2
        
    Args:
        seq1 (str) : First sequence being aligned
        seq2 (str) : Second sequence being aligned
        traceback_matrix (numpy array): traceback matrix
        maximum_position (tuple): starting position to trace back from
        
    Returns:
        aligned_seq1 (str): e.g. GTTGAC
        aligned_seq2 (str): e.g. GTT-AC
        
    Pseudocode:
        while current_move != END:
            current_move = traceback_matrix[current_row][current_col]
            if current_move == DIAG:
                ...
            elif current_move == UP:
                ...
            elif current_move == LEFT:
                ...
            
    '''
    # initialize a string tracking what our current move is
    curr_move = None
    # initialize tow empty lists, representing nucleotides from each sequence
    seq1_list = []
    seq2_list = []

    # initialize variables tracking row and column
    curr_row = maximum_position[0]
    curr_col = maximum_position[1]

    # enter a loop that ends when our current move is an end
    while curr_move != 0:
        # update what our current move is
        curr_move = traceback_matrix[curr_row][curr_col]
        # check if the current move is up, left, or diagonal
        if curr_move == 1:
            # DIAGONAL = 1
            # both sequences get their current nucleotide added to their lists
            seq1_list.append(seq1[curr_row - 1])
            seq2_list.append(seq2[curr_col - 1])
            # we move 1 position up and to the left
            curr_row -= 1
            curr_col -= 1

        elif curr_move == 2:
            # UP = 2
            # seq1 (vertical seq) has its current nuc added to its list
            seq1_list.append(seq1[curr_row - 1])
            # seq2 (horizontal seq) has a gap added to its list
            seq2_list.append("-")
            # move up one row
            curr_row -= 1
            
        elif curr_move == 3:
            # LEFT = 3
            # seq2 (horizontal seq) has its current nuc added to the list
            seq2_list.append(seq2[curr_col - 1])
            # seq1 (vertical seq) has a gap added
            seq1_list.append("-")
            # move left one column
            curr_col -= 1

    # reverse the lists of nucleotides and package as strings
    seq1_list.reverse()
    seq2_list.reverse()
    aligned1 = "".join(seq1_list)
    aligned2 = "".join(seq2_list)

    return aligned1, aligned2

In [4]:
def smith_waterman(seq1, seq2, match=1, mismatch=-1, gap=-1):
    '''Smith-Waterman algorithm for local alignment
    
    Args:
        seq1 (str): input seq 1
        seq2 (str): input seq 2
        match: default = +1
        mismatch: default = -1
        gap: default = -1
    
    Returns:
        aligned_seq1 (str)
        aligned_seq2 (str)
        score_matrix (numpy array): scoring matrix
    '''
    # create a shell graph with numrows = length of seq 1 + 1, numcols = length of seq 2 +1
    scoring_matrix = np.zeros(shape=(len(seq1) + 1, len(seq2) + 1), dtype=int)
    traceback_matrix = np.zeros(shape=(len(seq1) + 1, len(seq2) + 1), dtype=int)

    # initialize
    max_position = (0, 0)
    max_score = 0

    # calculate scores for all positions in the matrix
    for i in range(len(seq1) + 1):
        # skip the first row in the graph because those will all have scores of 0
        if i == 0:
            continue

        for j in range(len(seq2) + 1):
            # skip the first position in the column
            if j == 0:
                continue

            # score the current position
            curr_score, curr_traceback = cal_score(scoring_matrix, seq1, seq2, i - 1, j - 1, match, mismatch, gap)

            # update the current position on the matrix
            scoring_matrix[i][j] = curr_score
            traceback_matrix[i][j] = curr_traceback

            # update high score and position
            if curr_score > max_score:
                max_score = curr_score
                max_position = (i, j)

    # now all the matrices are scored
    # get the traceback from the position with the high score
    aligned1, aligned2 = traceback(seq1, seq2, traceback_matrix, max_position)

    # spit out the aligned sequences and the scoring matrix
    return aligned1, aligned2, scoring_matrix

In [ ]:
# Example from slides
seq1 = 'TACTTAG'
seq2 = 'CACATTAA'

aligned_seq1, aligned_seq2, score_matrix = smith_waterman(seq1,seq2)

print (aligned_seq1)
print (aligned_seq2)
print (score_matrix)
print(pd.DataFrame(score_matrix, columns=list("-"+seq2), index=list("-"+seq1)))

AC-TTA
ACATTA
[[0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 1 1 0 0]
 [0 0 1 0 1 0 0 2 1]
 [0 1 0 2 1 0 0 1 1]
 [0 0 0 1 1 2 1 0 0]
 [0 0 0 0 0 2 3 2 1]
 [0 0 1 0 1 1 2 4 3]
 [0 0 0 0 0 0 1 3 3]]
